# 0. Импорт и конфигурация

In [1]:
RANDOM_STATE = 42

import os
import math
import json
import numpy as np
import pandas as pd
from typing import List, Iterable, Tuple

def set_global_seed(seed: int = RANDOM_STATE) -> None:
    np.random.seed(seed)

set_global_seed(RANDOM_STATE)

# 1. Обработка данных

## Шаг 1. Считать данные

In [2]:
Student_ID = 31

In [3]:
datasets = [
    (
        'Give Me Some Credit',
        'https://www.kaggle.com/competitions/GiveMeSomeCredit/overview',
        'SeriousDlqin2yrs'
    ),
    (
        'Porto Seguro’s Safe Driver Prediction',
        'https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction/overview',
        'target'
    ),
    (
        'Statlog (Shuttle)',
        'https://archive.ics.uci.edu/dataset/148/statlog+shuttle',
        'class'
    ),
    (
        'HTRU2',
        'https://archive.ics.uci.edu/dataset/372/htru2',
        'class'
    ),
    (
        'Bank Marketing',
        'https://archive.ics.uci.edu/dataset/222/bank%2Bmarketing',
        'y'
    ),
]

dataset_id = None if Student_ID is None else Student_ID % len(datasets)
if dataset_id is None:
    print("ОШИБКА! Не указан порядковый номер студента в списке группы.")
else:
    print(f"Информация о датасете '{datasets[dataset_id][0]}' доступна по следующей ссылке: {datasets[dataset_id][1]}")
    print(f"Целевая переменная: {datasets[dataset_id][2]}")

Информация о датасете 'Porto Seguro’s Safe Driver Prediction' доступна по следующей ссылке: https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction/overview
Целевая переменная: target


Загрузите данные и считайте их в датафрейм

In [4]:
df = pd.read_csv('data/train.csv')
df.head()


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,...,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,...,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,...,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,...,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,...,3,1,1,3,0,0,0,1,1,0


## Шаг 2. Обработка данных

В обработке датасета вы имеете (почти) полную свободу (важно в итоге просто побить бейзлайн).

Что НЕОБХОДИМО сделать:
- обработать пропуски, если есть
- закодировать категориальные фичи

Что ПОЛЕЗНО сделать:
- удалить дубликаты, если есть
- обработать выбросы
- удалить лишние фичи, если есть (возможно, полезно будет посмотреть на корреляции числовых признаков)
- стандартизировать данные

ВАЖНО: обработанный по необходимым пунктам датафрейм запишите в переменную df_base, обработанный далее по вашему желанию датафрейм запишите в переменную df_processed.

НЕ ПЕРЕЗАПИСЫВАЙТЕ ЭТИ ПЕРЕМЕННЫЕ И df, иначе могут возникнуть проблемы с прохождением тестов

In [5]:
df_base = df.copy().dropna()
df_base.head()

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,...,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,...,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,...,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,...,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,...,3,1,1,3,0,0,0,1,1,0


In [6]:
df_processed = df_base.copy().drop_duplicates()

feature_columns = df_processed.columns.drop(['id', 'target'])

for feature in feature_columns:
    df_processed[feature] = (df_processed[feature] - df_processed[feature].mean()) / df_processed[feature].std()

df_processed.head()

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,0.050218,0.964585,0.213594,1.182226,-0.299997,-0.805892,1.700162,-0.442786,...,1.525424,-0.367358,1.255371,0.167831,-0.373505,0.769910,0.896916,-0.634729,-0.732225,2.349971
1,9,0,-0.453868,-0.540093,0.954361,-0.844891,-0.299997,-0.805892,-0.588178,2.258423,...,-1.046514,-0.367358,-1.104668,0.531911,-0.373505,0.769910,0.896916,-0.634729,1.365699,-0.425537
2,13,0,1.562475,3.973940,1.695129,1.182226,-0.299997,-0.805892,-0.588178,2.258423,...,-0.617858,0.463923,2.435391,-0.196249,-0.373505,0.769910,0.896916,-0.634729,1.365699,-0.425537
3,16,0,-0.957954,-0.540093,-0.897558,-0.844891,-0.299997,1.240859,-0.588178,-0.442786,...,-1.475170,0.463923,0.665362,0.531911,-0.373505,-1.298851,-1.114929,-0.634729,-0.732225,-0.425537
4,17,0,-0.957954,0.964585,-1.638325,1.182226,-0.299997,1.240859,-0.588178,-0.442786,...,-1.046514,-0.367358,-1.104668,-1.652567,-0.373505,-1.298851,-1.114929,1.575472,1.365699,-0.425537


## Шаг 3. Разделение на train/val/test


Разделите датафрейм на фичи и на целевую переменную.

In [7]:
X_base, y_base = df_base.iloc[:, 2:], df_base['target']
X_processed, y_processed = df_processed.iloc[:, 2:], df_processed['target']

Разделите датафреймы base и processed каждый на выборки train/val/test (запишите их в переменные ниже)

In [8]:
from sklearn.model_selection import train_test_split

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)


In [9]:
# Разделяем на train/test (80/20)
X_train_processed, X_test_processed, y_train_processed, y_test_processed = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)

# Для валидационной выборки - берем 25% от train (20% от всех данных)
X_train_processed, X_val_processed, y_train_processed, y_val_processed = train_test_split(
    X_train_processed, y_train_processed, test_size=0.25, random_state=42, stratify=y_train_processed
)

## Дополнительное задание*. Решение дисбаланса классов

Изучите ваш датасет на наличие дисбаланса классов. Постройте распределение классов таргет переменной.
Визуализируйте расположение классов через PCA.

Выберите стратегию, по которой будете компенсировать дисбаланс (undersampling, oversampling, интерполяция, генерация новых примеров). 
Подсказка: воспользуйтесь библиотекой `imblearn`.
Постройте визуализацию для сбалансированного датасета.

Рекомендуется далее в задании 3 сравнить различные модели на устойчивость к дисбалансу классов. Визуализация в любом виде приветствуется.

In [10]:

### BEGIN YOUR CODE

df_balanced = ...

### END YOUR CODE

# 2. Реализация метрик

Реализуйте метрики классификации с помощью `numpy`/`pandas`. В этом разделе запрещено использовать `sklearn.metrics`.

In [11]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def accuracy_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    """
    Реализовать accuracy = correct / total.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return np.mean(y_true == y_pred)

In [12]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def precision_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    """
    Реализовать precision = TP / (TP + FP).
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))

    if TP + FP == 0:
        return 0.0
    return TP / (TP + FP)

In [13]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def recall_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    """
    Реализовать recall = TP / (TP + FN).
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    if TP + FN == 0:
        return 0.0
    return TP / (TP + FN)

In [14]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def f1_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    """
    Реализовать F1 = 2 * P * R / (P + R).
    """
    P = precision_manual(y_true, y_pred)
    R = recall_manual(y_true, y_pred)

    if P + R == 0:
        return 0.0

    return (2 * P * R) / (P + R)

In [15]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def precision_recall_curve_manual(y_true: Iterable[int], y_score: Iterable[float]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Реализовать подсчет PR-кривой.
    Возвращать (precision_list, recall_list, thresholds) при варьировании порога.
    """
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    thresholds = np.sort(np.unique(y_score))

    precision_list = []
    recall_list = []

    for t in thresholds:
        y_pred = (y_score >= t).astype(int)

        p = precision_manual(y_true, y_pred)
        r = recall_manual(y_true, y_pred)

        precision_list.append(p)
        recall_list.append(r)

    return np.array(precision_list), np.array(recall_list), thresholds


In [16]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def roc_curve_manual(y_true: Iterable[int], y_score: Iterable[float]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Реализовать подсчет ROC-кривой.
    Возвращать (fpr_list, tpr_list, thresholds).
    """
    thresholds = np.sort(np.unique(y_score))

    fpr_list = []
    tpr_list = []

    for t in thresholds:
        y_pred = (y_score >= t).astype(int)

        tpr = recall_manual(y_true, y_pred)

        FP = np.sum((y_true == 0) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        fpr = FP / (FP + TN) if (FP + TN) != 0 else 0.0

        fpr_list.append(fpr)
        tpr_list.append(tpr)

    return np.array(fpr_list), np.array(tpr_list), thresholds

In [17]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def roc_auc_manual(fpr: Iterable[float], tpr: Iterable[float]) -> float:
    """
    Реализовать численную интеграцию по FPR (например, трапеции).
    """
    fpr = np.array(fpr)
    tpr = np.array(tpr)

    order = np.argsort(fpr)
    fpr_sorted = fpr[order]
    tpr_sorted = tpr[order]

    auc = np.trapz(tpr_sorted, fpr_sorted)
    return float(auc)


# 3. Классификация (sklearn)

Обучите различные алгоритмы классификации и сравните их между собой. В качестве baseline используйте логистическую регрессию.

## Шаг 1. Бейзлайн

In [18]:
### BEGIN YOUR CODE

from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(
    max_iter=1000,
    random_state=42
)
logreg.fit(X_train_base, y_train_base)

### END YOUR CODE

fh


## Шаг 2. KNN

Воспользуйтесь `sklearn.neighbors import KNeighborsClassifier`. Сравните с бейзлайном. 

In [19]:
### BEGIN YOUR CODE


### END YOUR CODE

## Шаг 3. Решающее дерево

Воспользуйтесь `sklearn.tree import DecisionTreeClassifier`. Сравните с бейзлайном. 

In [ ]:
### BEGIN YOUR CODE


### END YOUR CODE

## Шаг 4. Случайный лес

Воспользуйтесь `sklearn.ensemble import RandomForestClassifier`. Сравните с бейзлайном. 

In [ ]:
### BEGIN YOUR CODE


### END YOUR CODE

## Шаг 5. SVM

Воспользуйтесь `sklearn.svm import SVC`. Сравните с бейзлайном. 

In [ ]:
### BEGIN YOUR CODE


### END YOUR CODE

# Задание 4. Ансамблирование

Воспользуйтесь модулем `sklearn.ensemble` для реализации ансамблирования ранее обученных моделей.
Например, вы можете использовать `sklearn.ensemble.VotingClassifier`.
 Сравните с бейзлайном. 

In [ ]:
### BEGIN YOUR CODE


### END YOUR CODE

# Дополнительное задание**

Реализуйте в .py модуле (или в нескольких модулях, архитектура остается на ваше усматрение, будьте готовы ее объяснить):
- пайплайн обработки данных (как в задании 1), 
- пайплайн обучения/дообучения ансамбля (как в заданиях 2,3). Также вычислите метрики и выведите их в консоль 
- пайплайн для классификации нового объекта ансамблем моделей (без обучения этих моделей, только предсказание метрик)

